# Will hardly anyone bid? — TenderMining single-bidder model (v1)

**The product question:** the moment a German construction tender is published, can we tell that it will end with **0 or 1 bids**? For a contractor, such a lot means no price war and a very high chance of winning — but officially you only learn the bid count months later. This model predicts it on day one.

**The answer, measured on 3 months of tenders the model never saw (Apr–Jun 2026):**

| | out of 100 lots, how many end with 0–1 bids? |
|---|---|
| picking lots **at random** ("chance") | **17** |
| picking only lots **the model flags** | **37** |

A flagged lot is **2.2× more likely** to be a low-competition lot than a random one. The model is picky: it catches about **1 in 4** of all low-competition lots; the rest slip through. (Technical scores for comparison with the literature: PR-AUC 0.34 vs. 0.17 chance and 0.25 for a trade-code-only model; ROC-AUC 0.67.)

**How this notebook is organized — four parts:**
1. **Build the dataset** — inputs strictly from what was public on publication day; outcomes from the awards file.
2. **Train & compare against chance** — the model must beat random picking *and* a one-trick "trade code only" model.
3. **Trust checks** — four tripwires that would catch the model cheating (peeking at the future). All four pass.
4. **Bottom line & v2** — the numbers above in plain terms, and the one feature (buyer track record) expected to improve them most.

Every number in this header is printed by a cell below — nothing is quoted from outside the notebook.

## Part 1 — Build the dataset

First, install the machine-learning library (CatBoost — a standard choice for table-shaped data of this size) and the usual Python tools.

In [ ]:
# Setup: CatBoost + imports (CPU runtime is intentional — ~4k rows)
%pip install -q catboost
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
import catboost, sklearn
print('catboost', catboost.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__)

catboost 1.2.10 | pandas 2.2.2 | sklearn 1.6.1


**Load the data from Google Drive.** Two files: **tenders** = what each notice said on the day it was published (our inputs — ~18,800 notice versions), and **awards** = how each lot eventually ended, including the bid count (the answer sheet — ~5,400 decided lots).

In [ ]:
# Mount Google Drive and locate the two parquet extracts
from google.colab import drive
drive.mount('/content/drive')

import glob
TENDERS_NAME = 'tenders_cpv45_20260101-20260630.parquet'
AWARDS_NAME = 'awards_cpv45_20260101-20260630.parquet'
tenders_hits = glob.glob(f'/content/drive/MyDrive/**/{TENDERS_NAME}', recursive=True)
awards_hits = glob.glob(f'/content/drive/MyDrive/**/{AWARDS_NAME}', recursive=True)
print('tenders:', tenders_hits)
print('awards :', awards_hits)
assert len(tenders_hits) >= 1 and len(awards_hits) >= 1, 'parquet files not found in Drive'
TENDERS_PATH, AWARDS_PATH = tenders_hits[0], awards_hits[0]

Mounted at /content/drive
tenders: ['/content/drive/MyDrive/tenders_cpv45_20260101-20260630.parquet']
awards : ['/content/drive/MyDrive/awards_cpv45_20260101-20260630.parquet']


In [ ]:
# Load parquets; read the `role` tag embedded in each column's parquet metadata (FIELDS.md)
def load_with_roles(path):
    schema = pq.read_schema(path)
    roles = {}
    for name in schema.names:
        md = schema.field(name).metadata or {}
        roles[name] = md.get(b'role', b'').decode() or None
    df = pd.read_parquet(path)
    return df, roles

tenders, tender_roles = load_with_roles(TENDERS_PATH)
awards, award_roles = load_with_roles(AWARDS_PATH)

print('tenders rows:', len(tenders), '| cols:', tenders.shape[1])
print('awards rows :', len(awards), '| cols:', awards.shape[1])
print('tenders role counts:', pd.Series([r or 'MISSING' for r in tender_roles.values()]).value_counts().to_dict())
missing_roles = [c for c, r in tender_roles.items() if r is None]
print('columns without role metadata:', missing_roles)
# revision structure
key = ['procedure_id', 'lot_id']
print('unique tender lots:', tenders.groupby(key).ngroups)
print('revision count distribution:', tenders.groupby(key).size().value_counts().sort_index().to_dict())
print('unique award lots:', awards.groupby(key).ngroups)

tenders rows: 18813 | cols: 103
awards rows : 5387 | cols: 46
tenders role counts: {'categorical': 28, 'numeric': 20, 'bool': 14, 'plumbing': 9, 'date': 7, 'key': 6, 'hierarchical': 6, 'text': 5, 'nested': 4, 'entity': 4}
columns without role metadata: []
unique tender lots: 15414
revision count distribution: {1: 12970, 2: 1824, 3: 381, 4: 176, 5: 45, 6: 9, 7: 6, 8: 1, 9: 1, 10: 1}
unique award lots: 5356


**Build the answer sheet.** A lot counts as "low competition" if it ended with **0 or 1 bids**. That outcome is attached to every published version of the lot's notice. The firewall rule: the model's inputs come **only** from what was public on publication day — every other column from the awards file is dropped immediately, and an assertion fails the notebook if any slips through. Result: ~5,000 notice versions covering ~3,900 decided lots; about 1 in 10 ended with 0–1 bids.

In [ ]:
# Dataset assembly (TRAINING.md leakage rule 1: source firewall)
# 1) Latest award revision per lot supplies the label
aw = awards.sort_values('publication_date').groupby(key, as_index=False).tail(1).copy()

# 2) Drop reporting errors before anything else
def has_flag(flags, name):
    if flags is None: return False
    try: return name in list(flags)
    except TypeError: return False
bad = aw['quality_flags'].apply(lambda f: has_flag(f, 'winner_but_zero_tenders'))
print(f'dropping {bad.sum()} awards flagged winner_but_zero_tenders')
aw = aw[~bad]
aw = aw[aw['n_tenders'].notna()]

# 3) Label: insufficient competition = 0 or 1 bids
aw['label'] = (aw['n_tenders'] <= 1).astype(int)

# 4) Firewall: from awards keep ONLY join keys + label. Everything else is post-outcome.
aw_label = aw[key + ['label']].copy()

# 5) Every tender revision is a row; inner join attaches the lot's eventual label
data = tenders.merge(aw_label, on=key, how='inner')
awards_cols_leaked = [c for c in data.columns if c in set(awards.columns) - set(tenders.columns) - {'label'}]
assert awards_cols_leaked == [], f'awards columns leaked into features: {awards_cols_leaked}'

n_lots = data.groupby(key).ngroups
print(f'labeled rows (revisions): {len(data)} | labeled lots: {n_lots}')
lot_label = data.groupby(key)['label'].first()
print(f'lot-level base rate: {lot_label.mean():.4f} ({lot_label.sum()} positive lots)')

dropping 1 awards flagged winner_but_zero_tenders
labeled rows (revisions): 5016 | labeled lots: 3945
lot-level base rate: 0.1044 (412 positive lots)


**Turn each notice into 82 facts.** The model may look at: what is being bought (CPV trade codes), where (region, postal zone), who is buying (type of authority — municipality, state agency; never the buyer's *name*, which would let the model memorize), money (estimated value), timing (deadlines, project duration), and hurdles for bidders (bid bond, required certificates, exclusion grounds). Fields are selected *mechanically* by the role tag stored in the data files — nobody hand-picks convenient columns. Empty fields stay visibly empty ("no bid bond mentioned" is itself a clue).

In [ ]:
# Feature engineering (leakage rule 2: mechanical selection by role; unknown roles excluded)
# Defined as a function so the production dry-run (tripwire 4) reuses the exact same code.
NA = '__NA__'

def as_list(v):
    if v is None: return []
    if isinstance(v, (list, np.ndarray)): return list(v)
    if isinstance(v, float) and np.isnan(v): return []
    return [v]

def join_codes(v):
    vals = sorted({str(x) for x in as_list(v)})
    return '|'.join(vals) if vals else NA

def cat_str(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return NA
    return str(v)

def hier_levels(col):
    if 'cpv' in col: return [('cpv2', 2), ('cpv3', 3), ('cpv4', 4)]
    if 'nuts' in col: return [('nuts1', 3), ('nuts2', 4), ('nuts3', 5)]
    if 'postal' in col: return [('zone1', 1)]
    return None

# list-typed columns detected once on the full tenders frame, so any subset transforms identically
IS_LIST = {c: tenders[c].map(lambda v: isinstance(v, (list, np.ndarray))).any() for c in tenders.columns}

def build_features(df):
    Xf = pd.DataFrame(index=df.index)
    cats, nums, excl = [], [], []
    for col, role in tender_roles.items():
        s = df[col]
        if role == 'numeric':
            Xf[col] = pd.to_numeric(s, errors='coerce'); nums.append(col)
        elif role == 'bool':
            # 3-way categorical: missingness is informative (bid_bond_required null != False)
            Xf[col] = s.map(lambda v: NA if v is None or (isinstance(v, float) and np.isnan(v)) else str(bool(v)))
            cats.append(col)
        elif role == 'categorical':
            Xf[col] = s.map(join_codes) if IS_LIST[col] else s.map(cat_str)
            cats.append(col)
        elif role == 'hierarchical':
            levels = hier_levels(col)
            if levels is None:
                excl.append((col, 'hierarchical-unknown-scheme')); continue
            for lname, n in levels:
                new = f'{col}__{lname}'
                if IS_LIST[col]:
                    Xf[new] = s.map(lambda v, n=n: '|'.join(sorted({str(x)[:n] for x in as_list(v)})) or NA)
                else:
                    Xf[new] = s.map(lambda v, n=n: NA if v is None or (isinstance(v, float) and np.isnan(v)) else str(v)[:n])
                cats.append(new)
        elif role == 'date':
            if col == 'publication_date': continue  # the reference point, not a feature
            Xf[f'span__{col}'] = (pd.to_datetime(s) - pd.to_datetime(df['publication_date'])).dt.days
            nums.append(f'span__{col}')
        else:
            excl.append((col, role or 'MISSING'))
    return Xf, cats, nums, excl

X, cat_cols, num_cols, excluded = build_features(data)
assert 'buyer_name' in [c for c, _ in excluded], 'buyer_name must be excluded (rule 4)'
FEATURES = cat_cols + num_cols
print(f'features: {len(FEATURES)} ({len(cat_cols)} categorical, {len(num_cols)} numeric)')
print('excluded roles:', pd.Series([r for _, r in excluded]).value_counts().to_dict())

features: 82 (56 categorical, 26 numeric)
excluded roles: {'plumbing': 9, 'key': 6, 'text': 5, 'nested': 4, 'entity': 4}


**A guard against self-deception.** There are two ways to feed a category (like a region code) to a model: as a plain yes/no fact ("is this lot in Bavaria?") — harmless; or by replacing it with its average outcome ("Bavaria: 13% single-bid") — dangerous, because the feature is then built out of the answers. This cell makes sure CatBoost always uses the harmless way and can never silently switch to the dangerous one.

In [ ]:
# Cardinality check for leakage rule 4 (pure one-hot, no target statistics).
# Spec says one_hot_max_size=128 assuming all categoricals fit under it; the engineered
# prefix/combo columns exceed that (selection_criteria_types=650, cpv_additional__cpv4=438,
# nuts3 levels ~300+). Above one_hot_max_size CatBoost silently switches to target
# statistics — the exact thing rule 4 forbids — so we raise the knob to keep every
# column one-hot. The rule's substance (no target statistics) is preserved.
card = pd.Series({c: X[c].nunique() for c in cat_cols}).sort_values(ascending=False)
print(card.head(10))
ONE_HOT_MAX_SIZE = 1024
assert card.max() <= ONE_HOT_MAX_SIZE, 'raise ONE_HOT_MAX_SIZE: a categorical exceeds it -> CatBoost would use target statistics'
print(f'one_hot_max_size={ONE_HOT_MAX_SIZE} covers max cardinality {card.max()} -> all one-hot, no CTR')

selection_criteria_types        650
cpv_additional__cpv4            438
place_nuts3__nuts3              336
buyer_nuts__nuts3               300
cpv_additional__cpv3            210
exclusion_grounds               107
cpv_additional__cpv2             88
procurement_additional_types     68
platform_name                    50
place_nuts3__nuts2               39
dtype: int64
one_hot_max_size=1024 covers max cardinality 650 -> all one-hot, no CTR


**The honest exam setup.** We pretend today is **23 March 2026**: the model learns only from lots first published *before* that date, and is graded only on lots published *after* it — never seeing them during training. The code asserts that no lot ends up on both sides. Corrected notices count once per version, but each version of a 5-version lot only carries 1/5 of a vote, so heavily-corrected lots don't dominate.

In [ ]:
# Temporal group-aware split (leakage rule 3): a lot goes wholly to train or test
# by its FIRST publication_date; ~80/20; assert no lot straddles the boundary.
data['publication_date'] = pd.to_datetime(data['publication_date'])
first_pub = data.groupby(key)['publication_date'].transform('min')
lot_first = data.groupby(key)['publication_date'].min()
threshold = lot_first.quantile(0.8)
print('split threshold (80th pct of lot first publication):', threshold.date())

is_train = first_pub <= threshold
train_lots = set(map(tuple, data.loc[is_train, key].drop_duplicates().values))
test_lots = set(map(tuple, data.loc[~is_train, key].drop_duplicates().values))
assert not (train_lots & test_lots), 'a (procedure_id, lot_id) straddles the split boundary'

# 1/k revision weighting, k = the lot's revision count (train AND eval)
k = data.groupby(key)['label'].transform('size')
data['weight'] = 1.0 / k

y = data['label'].values
w = data['weight'].values
Xtr, ytr, wtr = X[is_train.values], y[is_train.values], w[is_train.values]
Xte, yte, wte = X[~is_train.values], y[~is_train.values], w[~is_train.values]
print(f'train: {len(Xtr)} rows / {len(train_lots)} lots | test: {len(Xte)} rows / {len(test_lots)} lots')
base_rate_test = np.average(yte, weights=wte)
print(f'weighted base rate — train: {np.average(ytr, weights=wtr):.4f} | test: {base_rate_test:.4f}')

split threshold (80th pct of lot first publication): 2026-03-23
train: 4052 rows / 3159 lots | test: 964 rows / 786 lots
weighted base rate — train: 0.0890 | test: 0.1667


## Part 2 — Train the model and compare it against chance

**What "chance" means here:** flag lots at random. Since ~17% of test lots ended with 0–1 bids, random flagging is right ~17% of the time — that's the number to beat, and it appears below as the "constant base" line (0.1667).

**Second opponent:** a one-trick model that only knows the trade code (CPV-4, e.g. "roofing works") and predicts each trade's historical rate. If our 82-fact model can't beat that, the notice details add nothing over just knowing the trade.

In [ ]:
# Train CatBoost (TRAINING.md model block) and evaluate against both baselines
from catboost import Pool

def make_model():
    return CatBoostClassifier(
        cat_features=cat_cols,
        one_hot_max_size=ONE_HOT_MAX_SIZE,  # pure one-hot, no target statistics (rule 4)
        auto_class_weights='Balanced',
        eval_metric='PRAUC',
        random_seed=42,
        verbose=False,
    )

train_pool = Pool(Xtr, ytr, weight=wtr, cat_features=cat_cols)
model = make_model()
model.fit(train_pool)

p_test = model.predict_proba(Xte)[:, 1]
pr_auc = average_precision_score(yte, p_test, sample_weight=wte)
roc_auc = roc_auc_score(yte, p_test, sample_weight=wte)

# Baseline (a): constant base-rate predictor -> PR-AUC equals the weighted base rate
pr_auc_const = average_precision_score(yte, np.full(len(yte), base_rate_test), sample_weight=wte)

# Baseline (b): single-feature cpv4 rate learned on train (weighted), applied to test
cpv4_tr = pd.DataFrame({'cpv4': Xtr['cpv_main__cpv4'], 'y': ytr, 'w': wtr})
rate = cpv4_tr.groupby('cpv4').apply(lambda g: np.average(g['y'], weights=g['w']), include_groups=False)
train_base = np.average(ytr, weights=wtr)
p_cpv4 = Xte['cpv_main__cpv4'].map(rate).fillna(train_base).values
pr_auc_cpv4 = average_precision_score(yte, p_cpv4, sample_weight=wte)
roc_auc_cpv4 = roc_auc_score(yte, p_cpv4, sample_weight=wte)

print(f'CatBoost      PR-AUC: {pr_auc:.4f} | ROC-AUC: {roc_auc:.4f}')
print(f'constant base PR-AUC: {pr_auc_const:.4f} (= weighted test base rate {base_rate_test:.4f})')
print(f'cpv4-only     PR-AUC: {pr_auc_cpv4:.4f} | ROC-AUC: {roc_auc_cpv4:.4f}')

CatBoost      PR-AUC: 0.3405 | ROC-AUC: 0.6703
constant base PR-AUC: 0.1667 (= weighted test base rate 0.1667)
cpv4-only     PR-AUC: 0.2539 | ROC-AUC: 0.6403


**How good is it — train vs test.** Two rows, four honest numbers per row:
- **Precision** = of the lots we flag, how many really ended with 0–1 bids.
- **Recall** = of all the lots that ended with 0–1 bids, how many we caught.

The *train* row will look near-perfect — that's memorization, and it's expected; ignore it. The **test** row is the only one that predicts real-world performance, because those lots never influenced the model.

In [ ]:
# Train vs test comparison (overfitting check).
# PR-AUC folds every precision/recall trade-off into one number; the @0.5 line shows
# one concrete operating point: of the lots the model flags, how many really were
# single-bid (precision), and how many of the real single-bid lots it caught (recall).
from sklearn.metrics import precision_score, recall_score

p_train = model.predict_proba(Xtr)[:, 1]
for name, yy, pp, ww in [('train', ytr, p_train, wtr), ('test ', yte, p_test, wte)]:
    pr = average_precision_score(yy, pp, sample_weight=ww)
    roc = roc_auc_score(yy, pp, sample_weight=ww)
    yhat = (pp >= 0.5).astype(int)
    prec = precision_score(yy, yhat, sample_weight=ww)
    rec = recall_score(yy, yhat, sample_weight=ww)
    base = np.average(yy, weights=ww)
    print(f'{name} | base rate {base:.3f} | PR-AUC {pr:.4f} | ROC-AUC {roc:.4f} | '
          f'@0.5 cut-off: precision {prec:.4f}, recall {rec:.4f}')

train | base rate 0.089 | PR-AUC 0.8936 | ROC-AUC 0.9925 | @0.5 cut-off: precision 0.6541, recall 0.9964
test  | base rate 0.167 | PR-AUC 0.3405 | ROC-AUC 0.6703 | @0.5 cut-off: precision 0.3712, recall 0.2494


**What drives the predictions.** The top-ranked fields should match common sense (trade, project duration, region, bid-bond hurdles) — and no single field should dominate. The calibration table asks: when the model says "30% risk", does it happen about 30% of the time? (Currently it over-states high risks — fine for *ranking* lots, would need a correction layer before quoting probabilities to customers.)

In [ ]:
# Feature importance + calibration + per-group sanity (TRAINING.md evaluation section)
# NB: index by model.feature_names_ (the Pool's column order), not the FEATURES list.
imp = pd.Series(model.get_feature_importance(train_pool), index=model.feature_names_).sort_values(ascending=False)
print('top 15 features:')
print(imp.head(15).round(2).to_string())

from sklearn.calibration import calibration_curve
frac_pos, mean_pred = calibration_curve(yte, p_test, n_bins=8, strategy='quantile')
print('\ncalibration (mean predicted -> observed positive fraction):')
for mp, fp in zip(mean_pred, frac_pos):
    print(f'  {mp:.3f} -> {fp:.3f}')

print('\nper-cpv3 observed single-bid rate vs mean prediction (test, groups with >=20 rows):')
grp = pd.DataFrame({'cpv3': Xte['cpv_main__cpv3'], 'y': yte, 'p': p_test, 'w': wte})
gg = grp.groupby('cpv3').apply(
    lambda g: pd.Series({'n': len(g), 'obs': np.average(g['y'], weights=g['w']), 'pred': np.average(g['p'], weights=g['w'])}),
    include_groups=False)
print(gg[gg['n'] >= 20].round(3).to_string())

top 15 features:
cpv_main__cpv4              7.13
duration_days               5.36
cpv_main__cpv3              4.58
place_nuts3__nuts2          3.31
bid_validity_raw            3.30
cpv_additional__cpv4        3.08
place_nuts3__nuts3          3.07
selection_criteria_types    2.85
bid_validity_days           2.66
span__period_end            2.64
span__period_start          2.63
cv_required                 2.48
buyer_activity              2.39
buyer_nuts__nuts2           2.37
cpv_additional__cpv3        2.35

calibration (mean predicted -> observed positive fraction):
  0.045 -> 0.066
  0.087 -> 0.083
  0.123 -> 0.107
  0.167 -> 0.158
  0.212 -> 0.150
  0.280 -> 0.174
  0.402 -> 0.217
  0.717 -> 0.322

per-cpv3 observed single-bid rate vs mean prediction (test, groups with >=20 rows):
          n    obs   pred
cpv3
450    98.0  0.134  0.161
451    80.0  0.063  0.074
452   397.0  0.191  0.349
453   218.0  0.242  0.238
454   171.0  0.090  0.174


## Part 3 — Trust checks (the four tripwires)

Models like this usually cheat by accident — some field quietly contains the answer, and the numbers look great until real money is on the line. Each check below is designed to catch one way of cheating.

**Check 1 — scrambled answers.** We deliberately shuffle the outcomes and retrain. Now there is nothing real to learn, so the score **must** collapse to chance level. If it stayed high, the pipeline itself would be feeding answers to the model.

In [ ]:
# Tripwire 1 — shuffled-label run: permute labels at LOT level (revisions of a lot keep
# a common, but randomly reassigned, label), retrain, evaluate on the untouched test set.
# With nothing to learn the score MUST collapse to the base rate.
rng = np.random.default_rng(42)
train_lot_ids = data.loc[is_train.values, key].apply(tuple, axis=1)
uniq = train_lot_ids.drop_duplicates().tolist()
lot2label = dict(zip(map(tuple, data[key].apply(tuple, axis=1)), data['label']))
orig = np.array([lot2label[l] for l in uniq])
shuffled = rng.permutation(orig)
shuf_map = dict(zip(uniq, shuffled))
ytr_shuf = train_lot_ids.map(shuf_map).values

model_shuf = make_model()
model_shuf.fit(Pool(Xtr, ytr_shuf, weight=wtr, cat_features=cat_cols))
p_shuf = model_shuf.predict_proba(Xte)[:, 1]
pr_shuf = average_precision_score(yte, p_shuf, sample_weight=wte)
roc_shuf = roc_auc_score(yte, p_shuf, sample_weight=wte)
print(f'shuffled-label PR-AUC: {pr_shuf:.4f} (base rate {base_rate_test:.4f}) | ROC-AUC: {roc_shuf:.4f} (chance 0.5)')
assert pr_shuf < base_rate_test * 1.5, 'TRIPWIRE: shuffled-label score did not collapse — pipeline leaks answers'
print('tripwire 1 PASSED: score collapsed to base rate')

shuffled-label PR-AUC: 0.1883 (base rate 0.1667) | ROC-AUC: 0.5414 (chance 0.5)
tripwire 1 PASSED: score collapsed to base rate


**Check 2 — too good to be true.** Published research on predicting competition at publication time tops out around ROC-AUC ≈ 0.7. A score far above that doesn't mean genius — it means a leak, until proven otherwise.

**Check 3 — one-field wonder.** Train a tiny model on each field *alone*. If any single field nearly matches the full model, that field probably encodes the outcome through a back door. Real signal here should be many weak clues, not one magic column.

In [ ]:
# Tripwire 2 — too-good alarm: literature tops out ~ROC-AUC 0.7 for call-time competition
# prediction; a near-perfect score means a leak until a specific feature is exonerated.
print(f'full-model ROC-AUC: {roc_auc:.4f}')
assert roc_auc < 0.85, 'TRIPWIRE: score too good to be true — hunt the leaking feature'
print('tripwire 2 PASSED: within plausible range\n')

# Tripwire 3 — single-feature audit: train on each feature alone; one feature scoring
# near the full model is suspicious (real signal here is many weak features).
# Constant features (homogeneous corpus, see Known caveats) cannot be trained on — skipped.
def single_feature_score(col):
    is_cat = col in cat_cols
    m = CatBoostClassifier(
        cat_features=[col] if is_cat else [],
        one_hot_max_size=ONE_HOT_MAX_SIZE,
        auto_class_weights='Balanced',
        iterations=200, random_seed=42, verbose=False,
    )
    m.fit(Pool(Xtr[[col]], ytr, weight=wtr, cat_features=[col] if is_cat else []))
    p = m.predict_proba(Xte[[col]])[:, 1]
    return average_precision_score(yte, p, sample_weight=wte)

constant = [c for c in FEATURES if Xtr[c].nunique(dropna=False) <= 1]
print('skipped as constant in train:', constant)
audit = pd.Series({c: single_feature_score(c) for c in FEATURES if c not in constant}).sort_values(ascending=False)
print('\ntop 12 single-feature PR-AUCs (full model: %.4f, base: %.4f):' % (pr_auc, base_rate_test))
print(audit.head(12).round(4).to_string())
suspicious = audit[audit > 0.9 * pr_auc]
print('\nfeatures within 90% of full model:', list(suspicious.index) if len(suspicious) else 'none')
print('tripwire 3', 'REVIEW NEEDED' if len(suspicious) else 'PASSED: no single feature rivals the full model')

full-model ROC-AUC: 0.6703
tripwire 2 PASSED: within plausible range

skipped as constant in train: ['notice_kind', 'contract_type', 'cpv_main__cpv2', 'procedure_languages', 'buyer_country', 'n_doc_references']

top 12 single-feature PR-AUCs (full model: 0.3405, base: 0.1667):
buyer_nuts__nuts3           0.2575
cpv_main__cpv4              0.2539
buyer_nuts__nuts1           0.2392
buyer_nuts__nuts2           0.2372
buyer_legal_type            0.2371
buyer_activity              0.2319
selection_criteria_types    0.2304
exclusion_grounds           0.2279
duration_days               0.2233
bid_validity_raw            0.2216
n_criteria_financial        0.2163
notice_subtype              0.2153

features within 90% of full model: none
tripwire 3 PASSED: no single feature rivals the full model


**Check 4 — dress rehearsal.** Score today's still-open tenders, where no outcome exists yet. If any model input could not be computed for them, that input secretly depended on the future — and the whole model would be unusable in practice. Passing means the model works on day one, which is the whole product.

In [ ]:
# Tripwire 4 — production dry-run: score still-open tenders (no award exists yet).
# Any feature that cannot be computed for them depended on the future.
awarded_keys = set(map(tuple, aw_label[key].values))
open_mask = ~tenders[key].apply(tuple, axis=1).isin(awarded_keys)
open_tenders = tenders[open_mask].copy()
print(f'still-open tender rows: {len(open_tenders)} ({open_tenders.groupby(key).ngroups} lots)')

X_open, cats_o, nums_o, _ = build_features(open_tenders)
assert cats_o + nums_o == FEATURES, 'feature set differs for open tenders — some feature depends on the future'
assert list(X_open.columns) == list(X.columns)
p_open = model.predict_proba(X_open)[:, 1]
print('all features computable for open tenders; scores produced for every row')
print('score distribution on open tenders:')
print(pd.Series(p_open).describe().round(3).to_string())
print(f'\nshare of open lots flagged above 0.5: {(p_open > 0.5).mean():.3f}')
print('tripwire 4 PASSED')

still-open tender rows: 13797 (11469 lots)
all features computable for open tenders; scores produced for every row
score distribution on open tenders:
count    13797.000
mean         0.242
std          0.186
min          0.002
25%          0.107
50%          0.190
75%          0.322
max          0.967

share of open lots flagged above 0.5: 0.102
tripwire 4 PASSED


## Part 4 — The bottom line, in buyer terms

Everything below is computed from the three unseen test months (April–June 2026). "Chance" means: you pick construction lots to chase at random. "Model" means: you only chase lots the model flags.

In [ ]:
# The bottom line — every sentence printed here is a fact about the unseen test months.
flag = p_test >= 0.5

chance = np.average(yte, weights=wte)                       # hit rate when picking at random
model_hit = np.average(yte[flag], weights=wte[flag])        # hit rate among the model's flags
caught = np.average(flag[yte == 1], weights=wte[yte == 1])  # share of single-bid lots we catch

print(f'BY CHANCE : out of 100 lots picked at random, {chance*100:.0f} end up with 0-1 bids.')
print(f'THE MODEL : out of 100 lots the model flags,  {model_hit*100:.0f} end up with 0-1 bids.')
print(f'          -> a flagged lot is {model_hit/chance:.1f}x more likely to be a low-competition lot than a random one.')
print(f'COVERAGE  : the model catches {caught*100:.0f} of every 100 low-competition lots (it is picky; the rest slip through).')

top = np.argsort(-p_test)[:50]
print(f'CONFIDENCE: among the 50 test lots the model was MOST sure about, {yte[top].mean()*100:.0f}% ended with 0-1 bids.')

BY CHANCE : out of 100 lots picked at random, 17 end up with 0-1 bids.
THE MODEL : out of 100 lots the model flags,  37 end up with 0-1 bids.
          -> a flagged lot is 2.2x more likely to be a low-competition lot than a random one.
COVERAGE  : the model catches 25 of every 100 low-competition lots (it is picky; the rest slip through).
CONFIDENCE: among the 50 test lots the model was MOST sure about, 32% ended with 0-1 bids.


## What v2 will do

The strongest known predictor is deliberately **not** in this model yet: **each buyer's own track record** — e.g. "this municipality's tenders end up single-bid 40% of the time". Buyers repeat their habits, so this one number is expected to move the hit rate more than anything already included.

It was left out of v1 because computing it honestly is subtle: for a tender published in March, we may only count outcomes that were **already public in March**. An award decided in January but only published in May was invisible at prediction time — counting it would be cheating (and would inflate our numbers). No off-the-shelf tool does this correctly; it needs a hand-built, date-aware calculation (TRAINING.md, leakage rule 5).

**The v2 plan:** add that one feature, rerun *exactly this notebook's exam* (same split, same checks), and compare the new bottom-line numbers against today's 37-vs-17. If the buyer-history feature works as the literature suggests, both the hit rate and the coverage should rise — and the trust checks in Part 3 will tell us if we accidentally cheated.